# 🇵🇦 Forecasting Beban Listrik Panama dengan Facebook Prophet

Notebook ini mengikuti alur **EDA → Pengecekan Data → Persiapan Time Series → Training → Forecast → Simpan Model** untuk data beban listrik nasional Panama (`nat_demand`) pada rentang 2015–2020.

## 1. Setup / Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from prophet import Prophet

DATA_PATH = r"C:\Users\hhfzsn\Downloads\Panama_beban_listrik_per_jam_2015-2020_.xlsx"
MODEL_PATH = "D:/FreeLLM/Open Code/Project/prophet_model.pkl"

## 2. Memuat Data

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="data")
deskripsi = pd.read_excel(DATA_PATH, sheet_name="deskripsi")

df.head()

In [ ]:
print("SHEETS:", df.sheet_names if hasattr(df, "sheet_names") else "n/a")
print("COLUMNS:", list(df.columns))
print("DTYPES:
", df.dtypes)
print("SHAPE:", df.shape)

In [ ]:
deskripsi

In [ ]:
df.isna().sum()

In [ ]:
df["nat_demand"].describe()

## 3. Exploratory Data Analysis (EDA) & Pengecekan Data

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])
print("RANGE WAKTU:", df["datetime"].min(), "sampai", df["datetime"].max())
print("JUMLAH HARI UNIK:", df["datetime"].dt.date.nunique())

In [ ]:
daily = (
    df.groupby(df["datetime"].dt.date)["nat_demand"]
    .sum()
    .reset_index()
    .rename(columns={"datetime": "ds", "nat_demand": "y"})
)
daily["ds"] = pd.to_datetime(daily["ds"])
daily.head()

In [ ]:
daily["y"].describe()

## 4. Persiapan Data untuk Prophet

In [ ]:
model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    interval_width=0.95,
)
model.fit(daily[["ds", "y"])

## 5. Forecast

In [ ]:
future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
fig = model.plot(forecast)
plt.show()

## 6. Simpan Model

In [ ]:
import pickle
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)
print("Model tersimpan:", MODEL_PATH)